In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from tueplots import bundles, cycler, figsizes
from tueplots.constants.color import palettes

plt.rcParams.update(bundles.icml2022())
plt.rcParams.update(figsizes.icml2022_full())
plt.rcParams.update(cycler.cycler(color=palettes.tue_plot))
plt.rcParams.update({"figure.dpi": 350})

In [ ]:
evaluation_result_path = "../data/newspaper_collection_evaluation_results_20_12_2025.csv"
df = pd.read_csv(evaluation_result_path)
df.info()

In [ ]:
# map emotions to integers
available_emotions = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "sad",
    "surprise",
    "neutral",
]
emotions_id_map = {e: i for i, e in enumerate(available_emotions)}
id_emotions_map = {i: e for i, e in enumerate(available_emotions)}

In [ ]:
# Some preprocessing
df["date"] = pd.to_datetime(df["date"])
df["dominant_emotion_id"] = df["dominant_emotion"].map(emotions_id_map)

### Subsample at a specific confidence level

In [ ]:
min_confidence = 0
total_df_length = len(df)
df = df.query("confidence > @min_confidence")
print(f"For confidence level '{min_confidence}' there are {len(df)}/{total_df_length} ({len(df)/total_df_length*100:.1f}%) entries remaining.")

In [ ]:
df.head(2)

In [ ]:
## earliest date they published 
newspapers = df["newspaper"].unique().tolist()
dates = []
for np in newspapers:
    first_publication = df.query("@np == newspaper").sort_values(by="date").reset_index(drop=True).loc[0, "date"]
    dates.append(first_publication)
    print(f"{np}: {first_publication}")
all_have_published = sorted(dates)[-2]

#### Group **dominant emotion** by different time intervalls (bins) and extract most dominant dominant emotion ☺

In [ ]:
dominant_emotion_df = df[
    ["name", "surname", "party", "newspaper", "dominant_emotion", "dominant_emotion_id", "date"] + available_emotions].copy()
dominant_emotion_df = dominant_emotion_df.reset_index(drop=True)
print(len(dominant_emotion_df), len(df)) 

In [ ]:
dominant_emotion_df[available_emotions] = 0
for i in range(len(dominant_emotion_df)):
    row = dominant_emotion_df.iloc[i]
    row_dominant_emotion = row["dominant_emotion"]
    row_dominant_emotion_id = emotions_id_map[row_dominant_emotion]
    dominant_emotion_df.loc[i, row_dominant_emotion] = 1

In [ ]:
dominant_emotion_df.head(3)

In [ ]:
grouped_df = dominant_emotion_df.groupby([pd.Grouper(key="date", freq="W"), "newspaper", "party"])[available_emotions].sum()
grouped_df = grouped_df.reset_index().query("date > @all_have_published").sort_values(by="date").reset_index(drop=True)
grouped_df.head(2)

In [ ]:
# pick dominant emotion as the one that appeared most frequently in the time interval
for i in range(len(grouped_df)):
    row = grouped_df.iloc[i]
    emotions = row[available_emotions]
    grouped_df.loc[i, "agg_dominant_emotion"] = emotions.argmax()
grouped_df

In [ ]:
for np in newspapers:
    if np == "taz": continue
    fig, ax = plt.subplots(1, 1)
    np = np
    ax.set_title(f"{np}")
    sns.scatterplot(
        data=grouped_df[grouped_df["newspaper"]==np].assign(
            emotion_label=lambda x: x["agg_dominant_emotion"].map(id_emotions_map)
        ),
        x="date",
        y="party",
        hue="emotion_label",
        ax=ax,
    )

    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0)

plt.show()

In [ ]:
def retrieve_emotion_over_time(df: pd.DataFrame, interval: int, selector="max", type: str = "dominant"):
    pass

## Compare overall dominant emotions

In [ ]:
agg_dominant_emotions_df = dominant_emotion_df.groupby(by=["party", "newspaper"])[available_emotions].sum().reset_index()
pivot_df = agg_dominant_emotions_df[agg_dominant_emotions_df["newspaper"]].pivot(index="newspaper", columns="party", values=available_emotions)

In [ ]:
rows, cols = 2, 3
fig, ax = plt.subplots(rows, cols)
for i in range(rows):
    for j in range(cols):
        idx = (i * cols) + j
        c_emotion = available_emotions[idx]
        agg_party_newspaper = np_coll_df.groupby(["party", "newspaper"], as_index=False)[c_emotion].mean()
        pivot_df = agg_party_newspaper.pivot(index="newspaper", columns="party", values=c_emotion)
        
        sns.heatmap(
            pivot_df, 
            ax=ax[i, j], 
            annot=True, 
            fmt=".2f", 
            # cmap="YlGnBu", 
            cbar=False,
            square=False,
            annot_kws={"fontsize": 6}
        )
        ax[i, j].set_title(c_emotion)
        ax[i, j].set_xlabel(None)
        ax[i, j].set_ylabel(None)
plt.savefig("fig/heatmap_emotion_by_party_newspaper.pdf")